### Project 3: Fine-Tuning an LLM with Low Rank Adaptation (LoRA).

In this project, you are asked to fine-tune and evaluate a transformer-based
language model called ELECTRA (Efficiently Learning an Encoder that Classifies
Token Replacements Accurately) to classify online customer service queries.

ELECTRA is a BERT-sized language model developed by Google Research that learns
language more efficiently than traditional masked language models by training
on a richer, token-level discrimination task. The dataset you will work with,
BANKING77, contains over 13,000 short customer service messages drawn from the
online banking domain — the kind of text a user might type into a bank's
support chatbot — each labeled with one of 77 fine-grained intent categories
such as identifying a failed transaction, requesting a replacement card, or
querying an exchange rate. The queries are short, realistic, and often subtly different
from one another, making accurate classification a genuine challenge for any
model. 

Rather than training ELECTRA from scratch, you will apply LoRA (Low-Rank
Adaptation), a parameter-efficient fine-tuning technique that updates only a
small number of additional weights while keeping the base model largely frozen
— significantly reducing the compute and memory costs of fine-tuning. You will
systematically explore the effect of key hyperparameters on model performance,
and track your experiments using Weights & Biases (W&B), a professional machine
learning experiment tracking platform. W&B automatically logs training metrics
such as loss and accuracy in real time, and provides an interactive dashboard
where you can visualize learning curves, compare runs across different
hyperparameter configurations, and identify where and why model performance
changes. You will use W&B to help interpret your training results and to 
draw meaningful conclusions about model behavior. Finally,
you will present your findings in a professional format suitable for a
technical audience. 

By the end of this project, you will have hands-on
experience with the full fine-tuning pipeline: from data preparation and
efficient adaptation of a pretrained transformer, through experimentation and
performance analysis, to clear and professional communication of results.

#### Resources

- PEFT Documentation: https://huggingface.co/docs/peft/
- Banking77 Paper: https://arxiv.org/abs/2003.04807
- ELECTRA Paper: https://arxiv.org/abs/2003.10555
- Wandb Tutorial: https://docs.wandb.ai/quickstart

In [1]:
#### Setup and Installation - Colab

# Uncomment and run if using Google Colab
# !pip install transformers datasets peft wandb scikit-learn
# !pip install torch torchvision torchaudio


Additional Notes:

If you are running on Colab, enable the GPU runtime via Runtime -> Change runtime type -> T4 GPU (or similar)

If you are running the model locally, create & activate a virtual environment using the requirements.txt file
included with the assignment materials.


In [2]:
#### SECTION 1: IMPORTS

import os
import getpass
import wandb
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import (classification_report,confusion_matrix)
from sklearn.model_selection import train_test_split

os.environ["WANDB_API_KEY"] = getpass.getpass("Paste NEW W&B API key: ")
login_ok = wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

print("Login OK:", login_ok)

/Users/magmac/Downloads/School/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/magmac/.netrc
wandb: Currently logged in as: michaelghattas (michaelghattas-ghattas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Login OK: True


In [3]:
# For Local Computer:  Set Device and Confirm GPU or CPU
# Comment this section out if using Colab

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print('Using device:', DEVICE)
print('PyTorch version:', torch.__version__)

Using device: mps
PyTorch version: 2.12.0


### SECTION 2: CONFIGURATION DICTIONARY 

In [4]:
# Initial balanced configuration for ELECTRA-base + LoRA on BANKING77.
# Run at least one additional experiment for Q5 by changing one CONFIG value
# such as lora_r=8 or learning_rate=2e-4.

CONFIG = {
    "model"         : "google/electra-base-discriminator",
    "batch_size"    : 16,
    "learning_rate" : 1e-4,
    "epochs"        : 6,
    "lora_r"        : 16,
    "lora_alpha"    : 32,
    "max_length"    : 128,
    "seed"          : 42,
    "wandb_project" : "banking77-lora-michael-project3",
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])


#### SECTION 3: WANDB SETUP FUNCTION

In [5]:
def setup_wandb(config: dict) -> None:
    """
    Authenticate with WandB and initialise a new experiment run.

    wandb.login() checks for cached credentials first. If none are found
    it prompts interactively — you only need to do this once.

    wandb.init() opens the run and logs the full CONFIG dict as the
    hyperparameter record for this experiment.
    """
    print("Connecting to WandB...")
    wandb.login()

    wandb.init(
        project = config["wandb_project"], 
        name    = (
            f"electra-lora"
            f"_r{config['lora_r']}"
            f"_alpha{config['lora_alpha']}"
            f"_lr{config['learning_rate']}"
            f"_ep{config['epochs']}"
        ),
        config = config,
    )
    print(f"  WandB run     : {wandb.run.name}")
    print(f"  Dashboard URL : {wandb.run.url}\n")

### SECTION 4: DATA LOADER FUNCTION

TODO: 

    Load the raw Banking77 dataset:

    Create these reference variables:
        - dataset
        - label_names (list[str])
        - num_labels (int)
        
    Print the following in f-string format:
        - raw number of Train data
        - raw number of Test data
        - number of labels
        - sample of label_names (intent categories)
    
    Return {dict}
        Keys -> train, test, num_labels, label_names
        


In [6]:
def load_data() -> dict:
    """
    Load BANKING77 dataset and recover label names robustly.

    Returns:
        dataset     — Hugging Face DatasetDict
        label_names — ordered list of intent category names
        num_labels  — number of labels
    """
    print("Loading Banking77 dataset...")

    dataset = load_dataset("mteb/banking77")

    label_feature = dataset["train"].features["label"]

    # Newer/alternate HF schema: label may be ClassLabel
    if hasattr(label_feature, "names") and label_feature.names is not None:
        label_names = list(label_feature.names)

    # Fallback: mteb/banking77 may expose labels only as numeric Value
    else:
        num_labels = len(set(dataset["train"]["label"]))
        label_names = [str(i) for i in range(num_labels)]

    num_labels = len(label_names)

    print(f"Train examples    : {len(dataset['train']):,}")
    print(f"Test examples     : {len(dataset['test']):,}")
    print(f"Number of labels  : {num_labels}")

    return {
        "dataset": dataset,
        "label_names": label_names,
        "num_labels": num_labels,
    }

### SECTION 5: LOAD TOKENIZER FUNCTION

In [7]:
def load_tokenizer(config: dict) -> AutoTokenizer:
    """
    Load the tokenizer for the configured model.
    Ensures a pad token is set, which is required for batch processing.

    Returns the configured tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(config["model"])

    # ELECTRA tokenizers normally already define [PAD]. This fallback keeps
    # the notebook robust if a different compatible checkpoint is used later.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.unk_token

    print(f"Tokenizer loaded: {config['model']}")
    print(f"Pad token       : {tokenizer.pad_token}")

    return tokenizer


### SECTION 6:TOKENIZATION FUNCTION

In [8]:
from sklearn.model_selection import train_test_split

def tokenize_data(raw_data, tokenizer, config):
    """
    Tokenize BANKING77 train/test data and create stratified train/validation/test splits.
    Removes string metadata columns such as label_text so Trainer only sees tensor-safe fields.
    """

    dataset = raw_data["dataset"]
    val_size = config.get("val_size", 0.10)
    seed = config.get("seed", 42)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=config["max_length"],
        )

    # Remove every original column except label after tokenization.
    train_remove_cols = [c for c in dataset["train"].column_names if c != "label"]
    test_remove_cols  = [c for c in dataset["test"].column_names if c != "label"]

    tokenized_train_full = dataset["train"].map(
        tokenize,
        batched=True,
        remove_columns=train_remove_cols,
    )

    tokenized_test = dataset["test"].map(
        tokenize,
        batched=True,
        remove_columns=test_remove_cols,
    )

    # Rename label column to labels for Hugging Face Trainer compatibility.
    tokenized_train_full = tokenized_train_full.rename_column("label", "labels")
    tokenized_test = tokenized_test.rename_column("label", "labels")

    labels = tokenized_train_full["labels"]
    indices = list(range(len(tokenized_train_full)))

    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_size,
        random_state=seed,
        stratify=labels,
    )

    train_data = tokenized_train_full.select(train_idx)
    val_data = tokenized_train_full.select(val_idx)
    test_data = tokenized_test

    train_data.set_format("torch")
    val_data.set_format("torch")
    test_data.set_format("torch")

    print(f"Tokenized train examples : {len(train_data):,}")
    print(f"Tokenized val examples   : {len(val_data):,}")
    print(f"Tokenized test examples  : {len(test_data):,}")
    print(f"Dataset columns          : {train_data.column_names}")

    return {
        "train": train_data,
        "val": val_data,
        "test": test_data,
    }

### SECTION 7: EVALUTATION METRICS

This is a Trainer callback. 

In [9]:
def compute_metrics(eval_pred) -> dict:
    """
    Evaluation callback passed to the Trainer.
    Converts raw logits to class predictions and computes accuracy and
    weighted precision, recall, and F1 via a single classification_report call.
    Called automatically after each evaluation step — and by the Trainer's
    WandB integration, which picks up whatever dict this function returns and
    logs it to the dashboard automatically.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    report         = classification_report(labels, predictions, output_dict  = True, zero_division= 0)
    weighted       = report["weighted avg"]
    
    return {
        "accuracy"           : report["accuracy"],
        "f1_weighted"        : weighted["f1-score"],
        "precision_weighted" : weighted["precision"],
        "recall_weighted"    : weighted["recall"],
    }


### SECTION 8: MODEL EVALUATION

In [10]:
def evaluate_model(model, dataset, tokenizer: AutoTokenizer, config: dict) -> dict:
    """
    Run inference on a dataset and return metrics and raw predictions.
    This function is reused identically for the baseline and all post-training
    evaluations — the model passed in is the only thing that changes.

    report_to is intentionally kept empty here because this function is used
    for both the pre-training baseline (before the WandB run opens) and for
    final evaluation. Metrics from final evaluation are logged explicitly
    via wandb.summary in Section 12.

    Returns a dict with keys:
        accuracy    — float
        f1_weighted — float
        predictions — np.ndarray of predicted class indices
        labels      — np.ndarray of true class indices
    """
    eval_trainer = Trainer(
        model            = model,
        args             = TrainingArguments(
            output_dir                 = "./tmp_eval",
            per_device_eval_batch_size = config["batch_size"],
            dataloader_pin_memory      = False,
            remove_unused_columns      = False,
            report_to                  = [],
        ),
        processing_class = tokenizer,
        compute_metrics  = compute_metrics,
    )

    output = eval_trainer.predict(dataset)

    return {
        "accuracy"   : output.metrics["test_accuracy"],
        "f1_weighted": output.metrics["test_f1_weighted"],
        "predictions": np.argmax(output.predictions, axis=1),
        "labels"     : output.label_ids,
    }

### SECTION 9: LORA MODEL SETUP FUNCTION

Some things to consider

(a) Why do we target "query", "key", and "value" specifically,rather than all layers in the model?
(b) Why is bias="none" the standard choice for LoRA?
(c) What is the relationship between lora_r and lora_alpha, and what happens if you set alpha much higher than 2*r? 

In [11]:
def create_lora_model(model, config: dict) -> tuple:
    """
    Wrap the base model with LoRA adapters and report parameter efficiency.
    Only the adapter weights are trainable; the original model weights are frozen.

    Returns:
        lora_model — the PEFT-wrapped model
        param_pct  — percentage of parameters that are trainable (float)
    """
    lora_config = LoraConfig(
        task_type      = TaskType.SEQ_CLS,
        r              = config["lora_r"],
        lora_alpha     = config["lora_alpha"],
        target_modules = ["query", "key", "value"],
        bias           = "none",
        lora_dropout   = 0.05,
    )
    lora_model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in lora_model.parameters())
    param_pct = 100 * trainable / total

    print("LoRA adapters applied:")
    print(f"  Target modules   : {list(lora_config.target_modules)}")
    print(f"  Rank (r)         : {lora_config.r}")
    print(f"  Alpha            : {lora_config.lora_alpha}")
    print(f"  Scaling alpha/r  : {lora_config.lora_alpha / lora_config.r:.2f}")
    print(f"  Trainable params : {trainable:,}  ({param_pct:.3f}% of {total:,} total)\n")

    return lora_model, param_pct


### SECTION 10: TRAINING ARGUMENTS FUNCTION 

In [12]:
# Detect the current hardware at runtime and returns the correct precision settings as a dictionary, so you don't have to hardcode them manually.

def get_precision_flags():
    # Colab / Windows / Linux — CUDA GPU available
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return {"bf16": True,  "fp16": False}  # Modern NVIDIA (A100, L4)
        else:
            return {"bf16": False, "fp16": True}   # Older NVIDIA (T4, V100)
    
    # macOS Apple Silicon — MPS available
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return {"bf16": False, "fp16": False}       # MPS: neither supported reliably
    
    # CPU fallback (Windows no-GPU, older Macs)
    else:
        return {"bf16": False, "fp16": False}

precision = get_precision_flags()

In [13]:
def build_training_args(config: dict) -> TrainingArguments:
    """
    Declare all training hyperparameters in one place.
    Keeping this separate from the training loop makes it straightforward to
    experiment with different configurations without touching the training logic.

    Returns a configured TrainingArguments object.
    """
    return TrainingArguments(
        output_dir                  = "./lora_banking_model",
        num_train_epochs            = config["epochs"],
        per_device_train_batch_size = config["batch_size"],
        per_device_eval_batch_size  = config["batch_size"],
        learning_rate               = config["learning_rate"],
        max_grad_norm               = 1.0,
        warmup_ratio                = 0.06,
        weight_decay                = 0.01,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        logging_steps               = 50,
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1_weighted",
        greater_is_better           = True,
        **precision,                # injects correct fp16/bf16 flags for platform, e.g., {"bf16": True, "fp16": False}
        dataloader_pin_memory       = False,
        remove_unused_columns       = False,
        seed                        = config["seed"],
        report_to                   = ["wandb"],   # <-- streams all metrics to WandB
    )


### SECTION 11: TRAINING LOOP FUNCTION

In [14]:
def run_training(
    model,
    data: dict,
    tokenizer: AutoTokenizer,
    training_args: TrainingArguments,
) -> Trainer:
    """
    Instantiate the Trainer with the prepared data and run the fine-tuning loop.
    The best checkpoint (by f1_weighted on the validation set) is reloaded
    automatically when training completes.

    Returns the trained Trainer instance.
    """
    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = data["train"], 
        eval_dataset     = data["val"],
        processing_class = tokenizer,
        compute_metrics  = compute_metrics,
    )

    print("Starting LoRA fine-tuning...")
    trainer.train()
    print("Training complete.\n")

    return trainer

### SECTION 12: ERROR ANALYSIS FUNCTION

In [15]:
def analyze_errors(predictions, labels, label_names: list) -> dict:
    """
    Surface two complementary views of model failure on the validation set.

    VIEW 1 — Confusion pairs (from the confusion matrix)
        Shows which specific intent pairs are most often mixed up.
        Answers: "Which two intents does the model confuse most?"
        This is useful for spotting localised, pairwise errors.

    VIEW 2 — Per-class precision, recall, and F1 (from classification_report)
        Shows where the model is weakest at the individual intent level.
        Precision answers: "When the model predicts intent X, how often is
            it correct?" — low precision means over-prediction of that intent.
        Recall answers: "Of all queries that are intent X, how many does the
            model catch?" — low recall means the model is missing that intent.
        F1 is the harmonic mean of both and is the primary sort key here.
        Together, precision and recall reveal whether a weak intent is caused
        by over-triggering, under-triggering, or both — a distinction that
        aggregate metrics like weighted F1 hide entirely.

    Returns a dict with keys:
        confusion_pairs     — list of (true_label, predicted_label, count) tuples,
                              sorted by count descending
        per_class           — list of (label, precision, recall, f1, support) tuples,
                              sorted by f1 ascending (weakest intents first)
        accuracy            — float, overall accuracy
        f1_weighted         — float, weighted F1
        precision_weighted  — float, weighted precision
        recall_weighted     — float, weighted recall
    """
    # --- View 1: confusion pairs ---
    cm    = confusion_matrix(labels, predictions)
    pairs = [
        (label_names[i], label_names[j], cm[i, j])
        for i in range(len(label_names))
        for j in range(len(label_names))
        if i != j and cm[i, j] > 0
    ]
    pairs.sort(key=lambda x: x[2], reverse=True)

    print("Top 10 Most Confused Intent Pairs (Validation Set):")
    print(f"{'True Intent':<36} {'Predicted As':<36} {'Count':>5}")
    print("-" * 79)
    for true_label, pred_label, count in pairs[:10]:
        print(f"{true_label:<36} {pred_label:<36} {count:>5}")

    # --- View 2: per-class precision, recall, F1 ---
    report   = classification_report(
        labels,
        predictions,
        target_names = label_names,
        output_dict  = True,
        zero_division= 0,
    )
    per_class = [
        (
            label,
            report[label]["precision"],
            report[label]["recall"],
            report[label]["f1-score"],
            int(report[label]["support"]),
        )
        for label in label_names
    ]
    per_class.sort(key=lambda x: x[3])   # ascending by F1 — weakest first

    print(f"\n10 Weakest Intents by F1 (Validation Set):")
    print(f"{'Intent':<36} {'Precision':>10} {'Recall':>8} {'F1':>8} {'N':>5}")
    print("-" * 69)
    for label, prec, rec, f1, n in per_class[:10]:
        print(f"{label:<36} {prec:>10.3f} {rec:>8.3f} {f1:>8.3f} {n:>5}")

    weighted = report["weighted avg"]
    return {
        "confusion_pairs"    : pairs,
        "per_class"          : per_class,
        "accuracy"           : report["accuracy"],
        "f1_weighted"        : weighted["f1-score"],
        "precision_weighted" : weighted["precision"],
        "recall_weighted"    : weighted["recall"],
    }

## Q5 Comparison Run Note

The main run used `lora_r=16`, `lora_alpha=32`, and produced the selected final model results summarized below. The following executable cell changes `lora_r` to 8 for the required Q5 comparison run. Therefore, if the notebook is run from this point forward, the output corresponds to the comparison run, while the main r=16 results are documented in the summary cells and W&B report.

In [16]:
# Q5 comparison run configuration
# Main run used lora_r = 16 and lora_alpha = 32.
# This comparison lowers rank to 8 while keeping alpha fixed.

wandb.finish()

CONFIG["lora_r"] = 8
CONFIG["lora_alpha"] = 32

print("Comparison run CONFIG updated:")
print("lora_r:", CONFIG["lora_r"])
print("lora_alpha:", CONFIG["lora_alpha"])
print("LoRA scaling alpha/r:", CONFIG["lora_alpha"] / CONFIG["lora_r"])

Comparison run CONFIG updated:
lora_r: 8
lora_alpha: 32
LoRA scaling alpha/r: 4.0


### SECTION 13: MAIN EXECUTION

In [17]:

# Step 1 — Connect to WandB
#   This must happen before training so the run is open when the
#   Trainer starts streaming metrics. The baseline is evaluated
#   before wandb.init() on purpose: it uses a bare Trainer with
#   report_to=[] (see evaluate_model), so no run needs to be open yet.

setup_wandb(CONFIG)

# Step 2 — Load and tokenize data

raw_data  = load_data()
tokenizer = load_tokenizer(CONFIG)
data      = tokenize_data(raw_data, tokenizer, CONFIG)

# Step 3 — Load base model and establish a pre-training baseline

print("Loading base ELECTRA model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model"],
    num_labels=raw_data["num_labels"],
).to(DEVICE)

print("Evaluating baseline (untrained) performance...")
baseline = evaluate_model(base_model, data["val"], tokenizer, CONFIG)
print(f"  Baseline Accuracy    : {baseline['accuracy']:.4f}")
print(f"  Baseline F1-Weighted : {baseline['f1_weighted']:.4f}\n")

# Log baseline to WandB so it appears alongside training metrics in the
# run summary. wandb.summary values are pinned to the run — they don't
# appear as time-series points but are always visible in the run overview.

wandb.summary["baseline_accuracy"]    = baseline["accuracy"]
wandb.summary["baseline_f1_weighted"] = baseline["f1_weighted"]

# Step 4 — Apply LoRA adapters to the base model

lora_model, param_pct = create_lora_model(base_model, CONFIG)
wandb.summary["trainable_param_pct"] = param_pct

# Step 5 — Configure and run fine-tuning
#   From this point onward the Trainer streams loss, learning rate, and
#   eval metrics to WandB automatically via report_to=["wandb"].

training_args = build_training_args(CONFIG)
trainer       = run_training(lora_model, data, tokenizer, training_args)

# Step 6 — Evaluate the fine-tuned model on validation and test sets

val_results  = evaluate_model(trainer.model, data["val"],  tokenizer, CONFIG)
test_results = evaluate_model(trainer.model, data["test"], tokenizer, CONFIG)

# Print the comparison table to the console
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"\n{'Metric':<22} {'Baseline':>10} {'Val (LoRA)':>12} {'Test (LoRA)':>12}")
print("-" * 58)
print(f"{'Accuracy':<22} {baseline['accuracy']:>10.4f} {val_results['accuracy']:>12.4f} {test_results['accuracy']:>12.4f}")
print(f"{'F1-Weighted':<22} {baseline['f1_weighted']:>10.4f} {val_results['f1_weighted']:>12.4f} {test_results['f1_weighted']:>12.4f}")
print(f"\n  Accuracy gain vs baseline : {val_results['accuracy'] - baseline['accuracy']:+.4f}")

gap  = val_results["accuracy"] - test_results["accuracy"]
flag = "✓ Good generalization" if abs(gap) < 0.02 else "⚠ Check for overfitting"
print(f"  Val → Test gap            : {gap:+.4f}  ({flag})")

# Log final evaluation metrics and the generalization gap to WandB summary.
# These join the baseline values logged earlier, giving a complete
# before/after picture on the run's overview page.

wandb.summary["val_accuracy"]         = val_results["accuracy"]
wandb.summary["val_f1_weighted"]      = val_results["f1_weighted"]
wandb.summary["test_accuracy"]        = test_results["accuracy"]
wandb.summary["test_f1_weighted"]     = test_results["f1_weighted"]
wandb.summary["accuracy_gain"]        = val_results["accuracy"] - baseline["accuracy"]
wandb.summary["val_test_gap"]         = gap

# Step 7 — Analyze the most common misclassifications

print()
error_analysis  = analyze_errors(
    val_results["predictions"],
    val_results["labels"],
    raw_data["label_names"],
)

# Log the single weakest intent to WandB summary as a concrete signal of
# where the model needs the most improvement.

weakest_label, weakest_prec, weakest_rec, weakest_f1, _ = error_analysis["per_class"][0]
wandb.summary["weakest_intent"]           = weakest_label
wandb.summary["weakest_intent_precision"] = weakest_prec
wandb.summary["weakest_intent_recall"]    = weakest_rec
wandb.summary["weakest_intent_f1"]        = weakest_f1

# Step 8 — Close the WandB run cleanly
#   wandb.finish() flushes any remaining buffered metrics, marks the run
#   as complete in the dashboard, and releases the connection. Without
#   this call the run stays in a "running" state until it times out.

wandb.finish()
print("WandB run closed.")

Connecting to WandB...


  WandB run     : electra-lora_r8_alpha32_lr0.0001_ep6
  Dashboard URL : https://wandb.ai/michaelghattas-ghattas/banking77-lora-michael-project3/runs/g4ho1qga

Loading Banking77 dataset...


Train examples    : 9,993
Test examples     : 3,076
Number of labels  : 77
Tokenizer loaded: google/electra-base-discriminator
Pad token       : [PAD]
Tokenized train examples : 8,993
Tokenized val examples   : 1,000
Tokenized test examples  : 3,076
Dataset columns          : ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Loading base ELECTRA model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 64826.45it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored w

Evaluating baseline (untrained) performance...


...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,Precision Weighted,Recall Weighted
1,3.391771,3.068930,0.278000,0.210669,0.252471,0.278000
2,2.012447,1.809092,0.496000,0.436833,0.494397,0.496000
3,1.584387,1.286889,0.665000,0.639872,0.695497,0.665000
4,1.232361,1.029077,0.746000,0.735436,0.761347,0.746000
5,1.041662,0.892448,0.773000,0.760008,0.790592,0.773000
6,0.997895,0.847257,0.791000,0.783129,0.797677,0.791000


Training complete.



FINAL RESULTS

Metric                   Baseline   Val (LoRA)  Test (LoRA)
----------------------------------------------------------
Accuracy                   0.0110       0.7910       0.7731
F1-Weighted                0.0006       0.7831       0.7662

  Accuracy gain vs baseline : +0.7800
  Val → Test gap            : +0.0179  (✓ Good generalization)

Top 10 Most Confused Intent Pairs (Validation Set):
True Intent                          Predicted As                         Count
-------------------------------------------------------------------------------
76                                   17                                       7
5                                    67                                       4
12                                   11                                       4
37                                   29                                       4
53                                   25                                       4
54                             

eval/accuracy,▁▄▆▇██
eval/f1_weighted,▁▄▆▇██
eval/loss,█▄▂▂▁▁
eval/precision_weighted,▁▄▇███
eval/recall_weighted,▁▄▆▇██
eval/runtime,█▁▃▂▃▃
eval/samples_per_second,▁█▆▇▆▆
eval/steps_per_second,▁█▆▇▆▆
train/epoch,▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
+3,...


WandB run closed.


Run name: electra-lora_r16_lr0.0001_ep6

batch_size: 16
learning_rate: 0.0001
epochs: 6
lora_r: 16
lora_alpha: 32
max_length: 128
LoRA scaling alpha/r: 2.00
trainable_param_pct: 1.382%

baseline_accuracy: 0.0180
baseline_f1_weighted: 0.0088

val_accuracy: 0.8050
val_f1_weighted: 0.7979

test_accuracy: 0.7971
test_f1_weighted: 0.7934

accuracy_gain: +0.7870
val_test_gap: +0.0079
weakest_intent: 66
weakest_intent_f1: 0.381

## Main Run Summary

The main run used ELECTRA-base with LoRA adapters applied to the query, key, and value projection modules. The selected configuration used batch size 16, learning rate 1e-4, 6 epochs, LoRA rank 16, LoRA alpha 32, and max sequence length 128. The effective LoRA scaling factor was alpha/r = 32/16 = 2.00.

The untrained ELECTRA baseline achieved 0.0180 validation accuracy and 0.0088 weighted F1. After LoRA fine-tuning, the model achieved 0.8050 validation accuracy, 0.7979 validation weighted F1, 0.7971 test accuracy, and 0.7934 test weighted F1. The validation-to-test gap was 0.0079, below the 0.02 threshold, indicating good generalization.

## Q5 Comparison Run Summary

Comparison run: `electra-lora_r8_alpha32_lr0.0001_ep6`

This run changed `lora_r` from 16 to 8 while keeping `lora_alpha = 32`, `learning_rate = 1e-4`, `epochs = 6`, and `max_length = 128`.

- LoRA rank: 8
- LoRA alpha: 32
- Effective scaling factor: 32 / 8 = 4.00
- Trainable parameters: 1,092,173
- Trainable parameter percentage: 0.987%

Results:

- Baseline accuracy: 0.0110
- Baseline weighted F1: 0.0006
- Validation accuracy: 0.7910
- Validation weighted F1: 0.7831
- Test accuracy: 0.7731
- Test weighted F1: 0.7662
- Accuracy gain: +0.7800
- Validation-to-test gap: +0.0179

Compared with the main run using `lora_r = 16`, this lower-rank run used fewer trainable parameters but produced slightly lower validation and test performance. The gap remained below 0.02, so the model still generalized acceptably.

## Main Run vs. Q5 Comparison Run

| Metric | Main Run: r=16, alpha=32 | Comparison Run: r=8, alpha=32 |
|---|---:|---:|
| LoRA scaling alpha/r | 2.00 | 4.00 |
| Trainable parameter % | 1.382% | 0.987% |
| Baseline accuracy | 0.0180 | 0.0110 |
| Baseline weighted F1 | 0.0088 | 0.0006 |
| Validation accuracy | 0.8050 | 0.7910 |
| Validation weighted F1 | 0.7979 | 0.7831 |
| Test accuracy | 0.7971 | 0.7731 |
| Test weighted F1 | 0.7934 | 0.7662 |
| Accuracy gain | +0.7870 | +0.7800 |
| Validation-test gap | +0.0079 | +0.0179 |

The `r=16` configuration performed better overall, especially on the held-out test set, while the `r=8` configuration was more parameter-efficient. Reducing the rank lowered the trainable parameter percentage from 1.382% to 0.987%, but it also reduced test weighted F1 from 0.7934 to 0.7662. This suggests that the higher-rank LoRA adapter had more useful adaptation capacity for the 77-class BANKING77 intent classification task.